# BoltzGen Universal Biomolecular Binder Design with `foldrun-shim`

This notebook adapts the official [**`HannesStark/boltzgen`**](https://github.com/HannesStark/boltzgen) ([`boltz-community/boltzgen`](https://huggingface.co/boltzgen/boltzgen-1)) design specification examples (`example/vanilla_protein/`, `example/vanilla_peptide_with_target_binding_site/`, `yaml_spec.md`) and public BoltzGen notebook workflows to run on **FoldRun**'s auto-scaling Google Cloud Vertex AI A100 Pipelines using **`foldrun-shim`**.

BoltzGen performs all-atom generative binder design and inverse folding across proteins, cyclic/linear peptides, nanobodies, antibodies, and small-molecule binding pockets using a multi-stage diffusion, inverse-folding, and Boltz-2 self-consistency ranking pipeline.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Examples & Design Specifications**:
  - Official [`HannesStark/boltzgen` Design Spec (`yaml_spec.md`)](https://github.com/HannesStark/boltzgen/blob/main/yaml_spec.md) and Example Workflows ([`example/vanilla_protein/`](https://github.com/HannesStark/boltzgen/tree/main/example/vanilla_protein), [`example/vanilla_peptide_with_target_binding_site/`](https://github.com/HannesStark/boltzgen/tree/main/example/vanilla_peptide_with_target_binding_site), [`example/small_molecule/`](https://github.com/HannesStark/boltzgen/tree/main/example/small_molecule), [`example/nanobody/`](https://github.com/HannesStark/boltzgen/tree/main/example/nanobody))
  - Official HuggingFace Model & Dataset Cards: [`boltzgen/boltzgen-1`](https://huggingface.co/boltzgen/boltzgen-1) and [`boltzgen/inference-data`](https://huggingface.co/datasets/boltzgen/inference-data)
- **Upstream Repository**: [`HannesStark/boltzgen`](https://github.com/HannesStark/boltzgen) (`boltz-community/boltzgen`)
- **Authors & Credits**: Hannes Stark, Gabriele Corso, Jeremy Wohlwend, Saro Passaro, Regina Barzilay, Tommi Jaakkola, and the **Boltz Community** (**MIT Jameel Clinic / CSAIL**).
- **License**: MIT License.
- **Primary Citations**:
  1. Stark, H., Corso, G., Wohlwend, J., Passaro, S., Barzilay, R., & Jaakkola, T. "BoltzGen: All-Atom Generative Biomolecular Binder Design." (2025). [github.com/HannesStark/boltzgen](https://github.com/HannesStark/boltzgen)
  2. Passaro, S., Corso, G., Wohlwend, J. *et al.* "Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction." *bioRxiv* (2025). [doi:10.1101/2025.06.14.659707](https://doi.org/10.1101/2025.06.14.659707)

---

## Why Run BoltzGen Through `foldrun-shim` Instead of a Standard Public Colab Notebook?

| Capability | Public BoltzGen Colab / Local CLI (`!boltzgen run ...`) | FoldRun Cloud Execution via `foldrun-shim` |
| :--- | :--- | :--- |
| **Model Checkpoints & Inference Data (~15+ GB)** | Downloads 5 large checkpoints (`boltzgen1_diverse.ckpt`, `boltzgen1_adherence.ckpt`, `boltzgen1_ifold.ckpt`, `boltz2_conf_final.ckpt`, `boltz2_aff.ckpt`) plus the `boltzgen/inference-data` dataset from HuggingFace on every fresh Colab session. | Pre-staged once on a shared **2.5 TB `BASIC_SSD` Cloud Filestore NFS share** at `/mnt/nfs/foldrun/boltzgen/cache` (`HF_HOME` and `BOLTZ_CACHE`), mounted into every worker container with zero cold-download latency. |
| **Multi-Seed GPU Topology** | Runs sequentially on a single Colab GPU (16 GB T4 frequently OOMs during joint diffusion + Boltz-2 self-consistency scoring). | 2-stage **`ConfigureSeeds -> ParallelFor[PredictOp]`** KFP topology fans out `--num_model_seeds` (`1..50`, default `5`) in parallel across dedicated **NVIDIA A100 40 GB** (`a2-highgpu-1g`, $\le 1500$ tokens) or **A100 80 GB** (`a2-ultragpu-1g`, $> 1500$ tokens) GPUs with DWS `FLEX_START`. |
| **Hotspot Indexing (`auth_seq_id` vs. `label_seq_id`)** | Upstream `boltzgen` strictly requires 1-indexed canonical mmCIF `label_seq_id` numbers and crashes or targets the wrong epitope when given crystallographic `auth_seq_id` numbers (e.g., `4Z18` where residues 1–17 are unresolved). | FoldRun's `target_resolver.py` and `predict_boltzgen` inspect the target `.cif` via `gemmi` and **automatically map crystallographic `auth_seq_id` hotspot numbers onto canonical 1-indexed `label_seq_id`** (`binding_types[*].chain.binding`). |
| **Target Structure Resolution & Path Handling** | Requires manual `.cif` downloads; passing absolute paths in `spec.yaml` causes upstream `boltzgen` to double-concatenate the working directory (`/tmp/work/tmp/work/input.cif`). | Supports **4-char RCSB PDB IDs** (`path: 4Z18`), **`gs://` URIs**, and **local `.cif`/`.pdb` files** (auto-bundled into `assets.tar.gz` by `foldrun-shim`), staging the target as a clean relative file alongside `query.yaml`. |
| **PyTorch 2.6+ & CLI Seed Compatibility** | Upstream `boltzgen` subprocesses fail on PyTorch 2.6+ (`weights_only=True` default in `torch.load`) and reject `--seed` on `boltzgen run`. | Automatically injects a `sitecustomize.py` shim (`torch.load(..., weights_only=False)`) into `PYTHONPATH` and seeds each parallel trajectory via `PL_GLOBAL_SEED=<seed_value>`. |
| **Notebook Disconnect Resilience** | Closing the browser tab or losing Wi-Fi terminates the design campaign and loses intermediate structures. | Persists `<out_dir>/.foldrun_job.json` upon `202 Accepted` submission. Re-running the cell re-attaches to the running Vertex AI `PipelineJob` and downloads `<stem>/final_ranked_designs/` and `<stem>/predictions/` when complete. |


## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")
os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")
# Optional: set FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT if delegating to an invoker service account
# os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = "foldrun-invoker@your-project.iam.gserviceaccount.com"

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ["FOLDRUN_GCS_BUCKET"])
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pyyaml",
    ],
    check=False,
)

import foldrun_shim
from foldrun_shim import boltzgen
from foldrun_shim.bundler import bundle_local_inputs

print(f"foldrun-shim version : {getattr(foldrun_shim, '__version__', 'unknown')}")
print(f"boltzgen CLI in PATH : {shutil.which('boltzgen') or 'available via foldrun_shim.cli'}")
print(f"FOLDRUN_GATEWAY_URL  : {os.environ['FOLDRUN_GATEWAY_URL']}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: "auto" (default), "L4" (fast & cost-effective), "A100" (40GB), or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "L4"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original Upstream Example (Unmodified / Verbatim Workflow)

### Original `HannesStark/boltzgen` Upstream Quickstart

Below is the exact quickstart command from the official [`HannesStark/boltzgen` repository](https://github.com/HannesStark/boltzgen):
```bash
!boltzgen run example/vanilla_protein/1g13prot.yaml --output workbench/test_run --protocol protein-anything --num_designs 10 --budget 2
```

We author `example/vanilla_protein/1g13prot.yaml` and execute the verbatim command below.


In [ ]:
# Write exact upstream HannesStark/boltzgen example/vanilla_protein/1g13prot.yaml
from pathlib import Path

spec_dir = Path("example/vanilla_protein")
spec_dir.mkdir(parents=True, exist_ok=True)
vanilla_spec = """entities:
  - protein:
      id: A
      sequence: 100..120
  - file:
      path: 1G13
"""
(spec_dir / "1g13prot.yaml").write_text(vanilla_spec, encoding="utf-8")
print("Wrote example/vanilla_protein/1g13prot.yaml")

In [ ]:
# Verbatim upstream BoltzGen quickstart execution:
!boltzgen run example/vanilla_protein/1g13prot.yaml --output workbench/test_run --protocol protein-anything --num_designs 10 --budget 2

---
## Part B — FoldRun Additions (`foldrun-shim` Python API, `/mnt/nfs/foldrun` Shared Databases & Weights, Local Asset Bundling, and Multi-GPU Seed Parallelism)

FoldRun supercharges BoltzGen with cloud enterprise features:
1. **Automatic 4-Character RCSB PDB Resolution (`path: 4Z18` / `path: 1G13`)**: Server-side resolution against `/mnt/nfs/foldrun/pdb_mmcif` so users never need to download local mmCIF files.
2. **Local Asset Bundling**: Automatically packages local target structures (`path: ./custom_target.cif`) into `assets.tar.gz`.
3. **Multi-Protocol Capabilities**: Full support for `nanobody-anything`, `antibody-anything`, and `protein-small_molecule` (`ccd: SAH`).
4. **Multi-Seed GPU Parallelism**: Runs up to 50 parallel seeds (`--num_model_seeds=5`) across Vertex AI A100 GPUs with DWS `FLEX_START`.
5. **Python Notebook Helper API (`boltzgen.predict`)**: Clean programmatic interface with status cards and downstream metrics analysis.


## 2. How the FoldRun Database (`/mnt/nfs/foldrun/boltzgen/cache`) & Target Resolver Work

### 2.1 Pre-Staged Unified NFS Cache (`boltzgen_cache` in `databases.yaml`)
BoltzGen combines a generative structure/sequence diffusion network, an inverse-folding model, and Boltz-2 structure/affinity verification models. Instead of downloading ~15+ GB from HuggingFace on every Colab session, FoldRun pre-populates `/mnt/nfs/foldrun/boltzgen/cache` on a **2.5 TB `BASIC_SSD` Cloud Filestore NFS share** via Cloud Batch (`HF_HUB_DISABLE_XET=1`):
- **Dataset**: `boltzgen/inference-data` (reference molecular & chemical component data).
- **5 Model Checkpoints (`boltzgen/boltzgen-1`)**:
  1. `boltzgen1_diverse.ckpt` — High-diversity all-atom backbone & sequence diffusion checkpoint.
  2. `boltzgen1_adherence.ckpt` — Condition-adherent diffusion checkpoint for strict hotspot/pocket constraints.
  3. `boltzgen1_ifold.ckpt` — Coupled inverse-folding sequence design checkpoint.
  4. `boltz2_conf_final.ckpt` — Boltz-2 complex co-folding & self-consistency (`sc_score`, `iptm`, `plddt`) checkpoint.
  5. `boltz2_aff.ckpt` — Boltz-2 binding affinity evaluation checkpoint.

At runtime, `predict_boltzgen` mounts `/mnt/nfs/foldrun` and exports `HF_HOME=/mnt/nfs/foldrun/boltzgen/cache` and `BOLTZ_CACHE=/mnt/nfs/foldrun/boltzgen/cache` while passing `--cache=/mnt/nfs/foldrun/boltzgen/cache` to `boltzgen check` and `boltzgen run`.

---

### 2.2 Three Target Structure & Design Modes
Depending on how you specify `entities:` in your BoltzGen YAML, `foldrun-shim` and FoldRun's `target_resolver.py` support three workflows:
1. **Mode 1 — 4-Character RCSB PDB ID (`path: 4Z18`) with Automatic `auth_seq_id -> label_seq_id` Mapping**:
   - Specify a 4-letter PDB code (`path: 4Z18` or `path: 6M1U`) directly in `entities[*].file.path`.
   - `foldrun-shim` recognizes the 4-character PDB ID and uploads only the YAML file.
   - Server-side `resolve_target_structure()` fetches `4z18.cif` from RCSB PDB, stages it to `gs://<bucket>/targets/<job_name>/4z18.cif`, and `map_auth_to_label_seq_id()` inspects `_atom_site.auth_seq_id` vs. `_atom_site.label_seq_id` via `gemmi` to translate crystallographic hotspot numbers (e.g., PD-L1 hotspots `54,56,66,115` in `4Z18`, where residues 1–17 are unresolved) onto canonical 1-indexed `label_seq_id` (`37,39,49,98`).
2. **Mode 2 — Local Custom Target Structure (`path: ./custom_target.cif`)**:
   - Reference any relative `.cif` or `.pdb` file in `entities[*].file.path`.
   - `foldrun_shim.bundler._collect_boltzgen_assets` validates path containment and bundles the structure into `assets.tar.gz`.
   - `foldrun-gateway` (`staging.py`) extracts the structure to `gs://<bucket>/gateway_jobs/<job_id>/inputs/assets/` and rewrites `file.path` so `predict_boltzgen` stages it alongside `query.yaml`.
3. **Mode 3 — Small-Molecule Binder Design (`protocol: protein-small_molecule`)**:
   - Design a *de novo* protein pocket around a small-molecule ligand specified via `ligand: {id: L, ccd: SAH}` or `smiles: ...`, resolved directly against the pre-staged NFS cache (`/mnt/nfs/foldrun/boltzgen/cache`).

---
## 3. Mode 1 (Interface 1): Drop-In Native CLI (`!boltzgen run ...`) with RCSB PDB Resolution (`4Z18`)

We first create `pdl1_binder.yaml` following the official `HannesStark/boltzgen` `yaml_spec.md` schema:
- **Designed Binder (`protein`)**: Chain `B` with variable length `sequence: 80..120`.
- **Target Structure (`file`)**: Human PD-L1 crystal structure `path: 4Z18`, Chain `A`, targeting the canonical PD-1 binding epitope (`binding: "54,56,66,115"`).

When you run `!boltzgen run pdl1_binder.yaml ...` in a notebook cell:
- `foldrun-shim` (`foldrun_shim.cli:boltzgen_main`) intercepts the command, strips local flags (`--output`, `--out_dir`, `--cache`, `--devices`, `--accelerator`), and forwards allowlisted scientific parameters (`--protocol protein-anything`, `--num_designs 4`, `--budget 2`, `--num_model_seeds 5`) to `foldrun-gateway`.
- On Vertex AI, `ConfigureSeeds` generates 5 independent random seeds and `ParallelFor[PredictOp]` launches **5 parallel NVIDIA A100 GPUs** (`5 seeds x 4 designs = 20 candidate designs`, retaining `5 x 2 = 10` diversity-filtered designs in the final budget).

In [ ]:
# Mode 1: BoltzGen YAML specification targeting RCSB PDB 4Z18 (Human PD-L1, Chain A)
# Note: Crystallographic residues 1-17 are unresolved in 4Z18, so auth_seq_id 54,56,66,115
# correspond to canonical 1-indexed label_seq_id 37,39,49,98. FoldRun maps these automatically via gemmi!
pdl1_binder_yaml = """entities:
  - protein:
      id: B
      sequence: 80..120
  - file:
      path: 4Z18
      include:
        - chain:
            id: A
      binding_types:
        - chain:
            id: A
            binding: "54,56,66,115"
"""

Path("pdl1_binder.yaml").write_text(pdl1_binder_yaml, encoding="utf-8")

# Verify that foldrun-shim recognizes '4Z18' as a server-resolved RCSB PDB ID (0 local files bundled)
bundled_mode1 = bundle_local_inputs(
    model="boltzgen",
    input_path="pdl1_binder.yaml",
    cli_args=[
        "--protocol",
        "protein-anything",
        "--num_designs",
        "4",
        "--budget",
        "2",
        "--num_model_seeds",
        "5",
    ],
)
print(
    "Wrote pdl1_binder.yaml (PDB target: 4Z18, Chain A hotspots: 54,56,66,115, Binder B: 80..120 aa)"
)
print(
    f"Bundled local assets (expected empty for 4-char PDB ID) : {list(bundled_mode1.assets_map.keys())}"
)
print(
    f"Deterministic client_input_sha256                       : {bundled_mode1.client_input_sha256}"
)

In [ ]:
# Interface 1: Drop-In Native BoltzGen CLI execution via foldrun-shim
# --output and --cache (if supplied) are stripped locally; 5 A100 GPUs run 4 designs (budget 2) each
!boltzgen run pdl1_binder.yaml --output ./boltzgen_out --protocol protein-anything --num_designs 4 --budget 2 --num_model_seeds 5

---
## 4. Mode 1 (Interface 2): Python Notebook Helper API (`boltzgen.predict`) & Interrupt Recovery

You can also submit and monitor BoltzGen campaigns directly from Python using `from foldrun_shim import boltzgen`:
- **Allowlisted Keyword Arguments (`_validate_boltzgen_flags`)**:
  - `protocol`: `"protein-anything"` (default), `"peptide-anything"`, `"protein-small_molecule"`, `"nanobody-anything"`, or `"antibody-anything"`.
  - `num_designs` (or `num_designs_per_seed`): `1..20` (default `2`) — number of diffusion designs generated per seed.
  - `budget` (or `budget_per_seed`): `1..20` (default `2`) — diversity-optimized budget retained per seed.
  - `num_model_seeds` (or `num_seeds`): `1..50` (default `5`) — number of parallel A100 GPU trajectories (`dsl.ParallelFor`).
  - `seed`: `int` (optional) — base random seed passed to `ConfigureSeeds`.
  - `binder_length_range` (`"80..140"`), `target_chains` (`"A"`), `binding_residues` (`"54,56,66,115"`).
- **Interactive HTML Card & Viewer Link**: `job` renders an HTML status card (`job._repr_html_()`) in Jupyter/Colab with a direct link to the FoldRun 3D Viewer (`job.viewer_url`).
- **Disconnect Recovery (`.foldrun_job.json`)**: If your Colab session disconnects while the A100 GPUs are running, re-running the cell reads `./boltzgen_out/.foldrun_job.json` and resumes polling the existing Vertex AI `PipelineJob`.

In [ ]:
# Interface 2: Python Notebook Helper API (5 A100 GPU seeds x 4 designs per seed, budget 2 per seed)
job = boltzgen.predict(
    "pdl1_binder.yaml",
    out_dir="./boltzgen_out",
    protocol="protein-anything",
    num_designs=4,
    budget=2,
    num_model_seeds=5,
    seed=42,
)

# Render interactive HTML status card with FoldRun 3D Viewer link
job

In [ ]:
# Inspect <out_dir>/.foldrun_job.json written by foldrun-shim for disconnect recovery
state_file = Path("./boltzgen_out/.foldrun_job.json")
if state_file.exists():
    state_data = json.loads(state_file.read_text(encoding="utf-8"))
    print("Saved interrupt-recovery state (.foldrun_job.json):")
    print(json.dumps(state_data, indent=2))
else:
    print("No .foldrun_job.json found yet (run the prediction cell above against a live gateway).")

---
## 5. Mode 2: Local Custom Target Structure (`path: ./custom_target.cif`) & Multi-Protocol Specifications

When designing binders against proprietary or newly solved experimental structures on your local disk (`path: ./custom_target.cif`), `foldrun-shim` automatically packages the local `.cif` or `.pdb` file:
1. `foldrun_shim.bundler._collect_boltzgen_assets` parses `entities[*].file.path` in your YAML, verifies that `./custom_target.cif` resides inside the workspace directory, and compresses it into `assets.tar.gz`.
2. `foldrun-gateway` (`staging.py`) extracts `assets.tar.gz` to `gs://<bucket>/gateway_jobs/<job_id>/inputs/assets/custom_target.cif` and rewrites `file.path` in the staged YAML.
3. `predict_boltzgen` stages `custom_target.cif` as a local relative file inside the container's working directory alongside `query.yaml` (avoiding upstream `boltzgen`'s double-path concatenation bug on absolute paths).

Below we demonstrate **Mode 2** with the `peptide-anything` protocol (adapted from `HannesStark/boltzgen/example/vanilla_peptide_with_target_binding_site/`), as well as reference specifications for `nanobody-anything` and `antibody-anything`.

In [ ]:
import json
import urllib.request
from pathlib import Path

# 1. Download or write local custom target structure (./custom_target.cif)
cif_path = Path("custom_target.cif")
if not cif_path.exists() or cif_path.stat().st_size < 10000:
    try:
        urllib.request.urlretrieve("https://files.rcsb.org/download/4Z18.cif", str(cif_path))
        print(f"Downloaded 4Z18.cif to {cif_path} ({cif_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"Failed to download 4Z18.cif ({e}), using inline fallback")
        custom_target_cif = """data_custom_target
#
_entry.id custom_target
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
_atom_site.auth_seq_id
_atom_site.auth_asym_id
ATOM 1  N N  ALA A 1 10.120 12.450  8.310 1.00 90.00 45 A
ATOM 2  C CA ALA A 1 11.420 12.980  8.720 1.00 90.00 45 A
ATOM 3  C C  ALA A 1 12.510 11.920  8.610 1.00 90.00 45 A
ATOM 4  O O  ALA A 1 12.310 10.740  8.320 1.00 90.00 45 A
ATOM 5  N N  TYR A 2 13.740 12.380  8.850 1.00 92.00 46 A
ATOM 6  C CA TYR A 2 14.890 11.490  8.790 1.00 92.00 46 A
ATOM 7  C C  TYR A 2 16.120 12.240  9.250 1.00 92.00 46 A
ATOM 8  O O  TYR A 2 16.050 13.450  9.480 1.00 92.00 46 A
ATOM 9  N N  GLU A 3 17.250 11.540  9.390 1.00 88.00 49 A
ATOM 10 C CA GLU A 3 18.510 12.150  9.810 1.00 88.00 49 A
ATOM 11 C C  GLU A 3 19.640 11.130  9.920 1.00 88.00 49 A
ATOM 12 O O  GLU A 3 19.420  9.930  9.750 1.00 88.00 49 A
#
"""
        cif_path.write_text(custom_target_cif, encoding="utf-8")
else:
    print(f"Using existing {cif_path} ({cif_path.stat().st_size} bytes)")

# 2. Write custom_peptide_binder.yaml referencing ./custom_target.cif (protocol: peptide-anything)
custom_peptide_yaml = """entities:
  - protein:
      id: B
      sequence: 14..22
  - file:
      path: ./custom_target.cif
      include:
        - chain:
            id: A
      binding_types:
        - chain:
            id: A
            binding: "45,46,49"
"""
Path("custom_peptide_binder.yaml").write_text(custom_peptide_yaml, encoding="utf-8")

# 3. Verify that foldrun_shim.bundler automatically collects custom_target.cif into assets.tar.gz
bundled_mode2 = bundle_local_inputs(
    model="boltzgen",
    input_path="custom_peptide_binder.yaml",
    cli_args=["--protocol", "peptide-anything", "--num_designs", "4", "--budget", "2"],
)
print(f"Bundled local target in assets.tar.gz : {list(bundled_mode2.assets_map.keys())}")
print(f"Archive size (compressed bytes)       : {len(bundled_mode2.assets_tar_bytes or b'')}")

# 4. Submit Mode 2 peptide binder design via boltzgen.predict
peptide_job = boltzgen.predict(
    "custom_peptide_binder.yaml",
    out_dir="./boltzgen_peptide_out",
    protocol="peptide-anything",
    num_designs=4,
    budget=2,
    num_model_seeds=3,
    seed=101,
)
peptide_job

In [ ]:
# Official-style BoltzGen specifications for 'nanobody-anything' and 'antibody-anything' protocols
nanobody_yaml = """entities:
  - protein:
      id: H
      sequence: 115..130
  - file:
      path: 4Z18
      include:
        - chain:
            id: A
      binding_types:
        - chain:
            id: A
            binding: "54,56,66,115"
"""
Path("nanobody_binder.yaml").write_text(nanobody_yaml, encoding="utf-8")

antibody_yaml = """entities:
  - protein:
      id: H
      sequence: 115..128
  - protein:
      id: L
      sequence: 105..115
  - file:
      path: 4Z18
      include:
        - chain:
            id: A
      binding_types:
        - chain:
            id: A
            binding: "54,56,66,115"
"""
Path("antibody_binder.yaml").write_text(antibody_yaml, encoding="utf-8")
print(
    "Wrote nanobody_binder.yaml (protocol: nanobody-anything) and antibody_binder.yaml (protocol: antibody-anything)"
)

# Example submission for nanobody design:
# nb_job = boltzgen.predict("nanobody_binder.yaml", out_dir="./boltzgen_nb_out", protocol="nanobody-anything", num_designs=4, budget=2, num_model_seeds=5)

---
## 6. Mode 3: Small-Molecule Binder Design (`protocol: protein-small_molecule`)

BoltzGen can also design *de novo* ligand-binding proteins around small molecules (`protocol: protein-small_molecule`) specified via a Chemical Component Dictionary code (`ccd: SAH`) or a `smiles` string (`ligand: {id: L, smiles: ...}`).

On FoldRun, chemical component definitions and inference data are resolved directly from the pre-staged NFS cache at `/mnt/nfs/foldrun/boltzgen/cache` (`boltzgen/inference-data`).

In [ ]:
# Mode 3: Designing a de novo protein binder (100..140 aa) around S-adenosyl-L-homocysteine (ccd: SAH)
small_molecule_yaml = """entities:
  - protein:
      id: B
      sequence: 100..140
  - ligand:
      id: L
      ccd: SAH
"""
Path("small_molecule_binder.yaml").write_text(small_molecule_yaml, encoding="utf-8")
print("Wrote small_molecule_binder.yaml (protocol: protein-small_molecule, ligand CCD: SAH)")

sm_job = boltzgen.predict(
    "small_molecule_binder.yaml",
    out_dir="./boltzgen_sm_out",
    protocol="protein-small_molecule",
    num_designs=4,
    budget=2,
    num_model_seeds=5,
    seed=42,
)
sm_job

---
## 7. Downstream Analysis, Multi-Seed Metrics Plotting & 3D Hotspot Visualization

When the Vertex AI `boltzgen-inference-pipeline` completes, `foldrun-gateway` (`output_packager.py`) collects all per-seed `.cif` and confidence `.json` files across the parallel A100 workers, ranks them globally by `ranking_score`, and materializes both the native BoltzGen `final_ranked_designs/` directory and per-seed `predictions/` directory inside `--output` / `out_dir`:

```text
boltzgen_out/
├── .foldrun_job.json
└── pdl1_binder/
    ├── final_ranked_designs/
    │   ├── metrics.csv
    │   ├── ranked_0.cif
    │   ├── ranked_1.cif
    │   └── ...
    ├── ranked_designs/
    │   ├── metrics.csv
    │   └── ranked_0.cif
    └── predictions/
        ├── design_seed1824901_0.cif
        ├── confidence_design_seed1824901_0.json
        └── ...
```

Below we load `<out_dir>/pdl1_binder/final_ranked_designs/metrics.csv` (and `<out_dir>/pdl1_binder/predictions/confidence_design_seed*_*.json`), plot `ranking_score`, `iptm`, `plddt`, and `sc_score` (self-consistency) across seeds with `matplotlib`, and render the top-ranked binder + target complex (`ranked_0.cif`) in `py3Dmol` with hotspot residues highlighted.

In [ ]:
# 7.1 Load final_ranked_designs/metrics.csv and per-seed confidence_design_seed*_*.json
stem_dir = Path("./boltzgen_out/pdl1_binder")
metrics_csv_path = stem_dir / "final_ranked_designs" / "metrics.csv"
predictions_dir = stem_dir / "predictions"

records = []
if metrics_csv_path.exists():
    with metrics_csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            records.append(
                {
                    "design_id": row.get("design_id") or row.get("design") or "design",
                    "seed_value": int(float(row.get("seed_value", 0) or 0)),
                    "ranking_score": float(row.get("ranking_score", 0.0)),
                    "iptm": float(row.get("iptm", 0.0)),
                    "plddt": float(row.get("plddt", row.get("mean_plddt", 0.0))),
                    "sc_score": float(row.get("sc_score", 0.0)),
                    "clash_score": float(row.get("clash_score", 0.0)),
                    "designed_sequence": row.get("designed_sequence", ""),
                }
            )
elif predictions_dir.exists():
    for jf in sorted(predictions_dir.glob("confidence_design_seed*.json")):
        c = json.loads(jf.read_text(encoding="utf-8"))
        records.append(
            {
                "design_id": c.get("design_id", jf.stem),
                "seed_value": int(c.get("seed_value", 0)),
                "ranking_score": float(c.get("ranking_score", 0.0)),
                "iptm": float(c.get("iptm", 0.0)),
                "plddt": float(c.get("plddt", c.get("mean_plddt", 0.0))),
                "sc_score": float(c.get("sc_score", 0.0)),
                "clash_score": float(c.get("clash_score", 0.0)),
                "designed_sequence": c.get("designed_sequence", ""),
            }
        )
    records.sort(key=lambda r: r["ranking_score"], reverse=True)

if records:
    print(f"Loaded {len(records)} ranked BoltzGen binder designs from {stem_dir}:\n")
    print(
        f"{'Rank':<6} {'Design ID':<26} {'Seed':>12} {'Rank Score':>11} {'ipTM':>8} {'pLDDT':>8} {'sc_score':>9} {'Clash':>7}"
    )
    print("-" * 94)
    for idx, r in enumerate(records):
        print(
            f"#{idx:<5} {r['design_id']:<26} {r['seed_value']:>12} "
            f"{r['ranking_score']:>11.4f} {r['iptm']:>8.4f} "
            f"{r['plddt']:>8.2f} {r['sc_score']:>9.4f} {r['clash_score']:>7.2f}"
        )
    if records[0].get("designed_sequence"):
        print(
            f"\nTop-Ranked Designed Binder Sequence (ranked_0.cif):\n  {records[0]['designed_sequence']}"
        )
else:
    print(
        f"No materialized outputs in {stem_dir} yet. Run Section 3 or 4 against a live FoldRun Gateway."
    )

In [ ]:
# 7.2 Plot ranking_score, iptm, plddt, and sc_score (self-consistency) across seeds with matplotlib
import matplotlib.pyplot as plt
import numpy as np

if records:
    labels = [r["design_id"].replace("design_", "") for r in records]
    ranking_scores = [r["ranking_score"] for r in records]
    iptms = [r["iptm"] for r in records]
    plddts = [r["plddt"] for r in records]
    sc_scores = [r["sc_score"] for r in records]

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    x = np.arange(len(records))

    # Panel 1: Overall Ranking Score
    axes[0, 0].bar(x, ranking_scores, color="#1a73e8", edgecolor="black", alpha=0.85)
    axes[0, 0].set_title("BoltzGen Composite Ranking Score Across Seeds")
    axes[0, 0].set_ylabel("ranking_score")
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    axes[0, 0].set_ylim(0, 1.05)

    # Panel 2: Interface pTM (ipTM)
    axes[0, 1].bar(x, iptms, color="#34a853", edgecolor="black", alpha=0.85)
    axes[0, 1].axhline(
        0.75,
        color="red",
        linestyle="--",
        linewidth=1,
        label="High-Confidence Threshold (0.75)",
    )
    axes[0, 1].set_title("Interface pTM (ipTM) Across Seeds")
    axes[0, 1].set_ylabel("iptm")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    axes[0, 1].set_ylim(0, 1.05)
    axes[0, 1].legend(loc="lower right", fontsize=8)

    # Panel 3: Mean pLDDT
    axes[1, 0].bar(x, plddts, color="#fbbc04", edgecolor="black", alpha=0.85)
    axes[1, 0].axhline(80.0, color="red", linestyle="--", linewidth=1, label="pLDDT >= 80")
    axes[1, 0].set_title("Predicted Local Distance Difference Test (pLDDT)")
    axes[1, 0].set_ylabel("plddt (0-100)")
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].legend(loc="lower right", fontsize=8)

    # Panel 4: Self-Consistency Score (sc_score) vs. ipTM scatter
    sc_plot = axes[1, 1].scatter(
        sc_scores, iptms, c=plddts, cmap="viridis", s=90, edgecolors="black"
    )
    axes[1, 1].set_title("Self-Consistency (sc_score) vs. Interface pTM")
    axes[1, 1].set_xlabel("sc_score (Self-Consistency)")
    axes[1, 1].set_ylabel("iptm")
    plt.colorbar(sc_plot, ax=axes[1, 1], label="pLDDT")

    plt.tight_layout()
    plt.show()
else:
    print("Metrics not materialized yet; run boltzgen.predict(...) first.")

In [ ]:
# 7.3 Interactive 3D Visualization of Top Designed Binder + Target Complex (ranked_0.cif) with py3Dmol
import py3Dmol

ranked_0_cif = Path("./boltzgen_out/pdl1_binder/final_ranked_designs/ranked_0.cif")
if not ranked_0_cif.exists():
    ranked_0_cif = Path("./boltzgen_out/pdl1_binder/ranked_designs/ranked_0.cif")

if ranked_0_cif.exists():
    cif_data = ranked_0_cif.read_text(encoding="utf-8")
    view = py3Dmol.view(width=840, height=560)
    view.addModel(cif_data, "cif")

    # Style Target Receptor (Chain A) in slate gray cartoon + translucent molecular surface
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#9aa0a6"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.35, "color": "#dadce0"}, {"chain": "A"})

    # Highlight Target Epitope Hotspot Residues on Chain A (auth_seq_id 54,56,66,115 / label_seq_id 37,39,49,98)
    hotspot_sel = {"chain": "A", "resi": [37, 39, 49, 98, 54, 56, 66, 115]}
    view.addStyle(hotspot_sel, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.25}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.65, "color": "#e91e63"}, hotspot_sel)

    # Style Designed Binder (Chain B) colored by per-residue pLDDT in B-factor column
    view.setStyle(
        {"chain": "B"},
        {
            "cartoon": {
                "colorscheme": {
                    "prop": "b",
                    "gradient": "roygb",
                    "min": 50,
                    "max": 95,
                }
            }
        },
    )

    view.zoomTo()
    view.show()
else:
    print(f"Top-ranked design structure {ranked_0_cif} not materialized yet.")

if "job" in globals() and getattr(job, "viewer_url", None):
    print(f"Full interactive FoldRun 3D Viewer URL: {job.viewer_url}")